# convexity.ipynb

- Reference: https://aubreymoore.github.io/jb/create-one-to-many-sql/

## TODO

- [x] move `get_data_for_images_table` function to `roadside.py`
- [x] move `get_data_for_detections_table` function to `roadside.py`
- [x] remove detection attribute fields from schema
- [x] prevent duplication of records in database
- [x] add fields to `detections` table
- [] handle images which do not contain detected objects
- [x] change filename of this notebook
- [] segmentation results saved in `~Desktop/blog2026` - why??
- [] find second weights file

In [1]:
%matplotlib widget

import roadside as rs
import sqlite3
import os
from icecream import ic
import pandas as pd
import math
import numpy as np
from shapely.wkt import loads as wkt_loads
import cv2
import textwrap
import gc
import torch


# reference: https://kioku-space.com/en/jupyter-skip-execution/
from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
    return

# Functions

# Global variables for this run

In [3]:
root_dir = "/home/aubrey/Desktop/sam3-2026-01-31"
image_paths = ["example_images/08hs-palms-03-zglw-superJumbo.webp",
               "example_images/20251129_152106.jpg"]
text_prompts=["coconut palm tree"]
db_path = 'test.db'

# Main

In [4]:
if not os.path.exists(db_path):
    rs.build_db(db_path, image_paths)


Database test.db created/opened.
SQL script executed successfully.
Changes committed.

Database connection closed.
Ultralytics 8.4.33 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Resolved 37 packages in 10.08s
   Updating https://github.com/ultralytics/CLIP.git (HEAD)
    Updated https://github.com/ultralytics/CLIP.git (88ade288431a46233f1556d1e141901b3ef0a36b)
   Building clip @ git+https://github.com/ultralytics/CLIP.git@88ade288431a46233f1556d1e141901b3ef0a36b
      Built clip @ git+https://github.com/ultralytics/CLIP.git@88ade288431a46233f1556d1e141901b3ef0a36b
Prepared 4 packages in 15.15s
Installed 4 packages in 0.98ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@88ade288431a46233f1556d1e141901b3ef0a36b)
 + ftfy==6.3.1
 + regex==2026.3.32
 + tqdm==4.67.3

requirements: AutoUpdate success ✅ 25.3s
WA

ic| 'copying results_gpu to results_cpu'
ic| 'deleting results_gpu from GPU and clearing caches'


Ultralytics 8.4.33 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)


ic| df_input:                                           image_path  detection_id  \
              0  example_images/08hs-palms-03-zglw-superJumbo.webp             1   
              1  example_images/08hs-palms-03-zglw-superJumbo.webp             2   
              
                                                          poly_wkt  \
              0  POLYGON ((486 679, 486 684, 485 685, 485 688, ...   
              1  POLYGON ((1035 1274, 1035 1276, 1034 1277, 103...   
              
                                               hull_indices_string  
              0  1067 1005 949 939 938 898 729 579 575 573 571 ...  
              1  1006 1004 1003 1001 995 855 670 668 664 654 32...  



image 1/1 /home/aubrey/Desktop/temp/example_images/20251129_152106.jpg: 1932x1932 25 coconut palm trees, 1221.1ms
Speed: 10.9ms preprocess, 1221.1ms inference, 2.8ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict80


ic| 'copying results_gpu to results_cpu'
ic| 'deleting results_gpu from GPU and clearing caches'


Error occurred while calculating convexity defects for row 5: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/convhull.cpp:397: error: (-5:Bad argument) The convex hull indices are not monotonous, which can be in the case when the input contour contains self-intersections in function 'convexityDefects'

Error occurred while calculating convexity defects for row 10: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/convhull.cpp:397: error: (-5:Bad argument) The convex hull indices are not monotonous, which can be in the case when the input contour contains self-intersections in function 'convexityDefects'

Error occurred while calculating convexity defects for row 14: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/convhull.cpp:397: error: (-5:Bad argument) The convex hull indices are not monotonous, which can be in the case when the input contour contains self-intersections in function 'convexityDefects'

Error occurred while calculating convexity defects for row 18: OpenCV(4.13.0) /io/opencv/

ic| df_input:                                            image_path  detection_id  \
              0   example_images/08hs-palms-03-zglw-superJumbo.webp             1   
              1   example_images/08hs-palms-03-zglw-superJumbo.webp             2   
              2                  example_images/20251129_152106.jpg             3   
              3                  example_images/20251129_152106.jpg             4   
              4                  example_images/20251129_152106.jpg             5   
              5                  example_images/20251129_152106.jpg             6   
              6                  example_images/20251129_152106.jpg             7   
              7                  example_images/20251129_152106.jpg             8   
              8                  example_images/20251129_152106.jpg             9   
              9                  example_images/20251129_152106.jpg            10   
              10                 example_images/20251129_152106.j

In [3]:
assert rs.check_gpu(), 'ERROR: GPU is unavailable.'

rs.create_db(db_path)
con = sqlite3.connect(db_path)

# scan images and populate database
for image_path in image_paths:
    
    # skip image if it is already in the database
    if con.execute(f'SELECT COUNT(*) FROM images WHERE  image_path = "{image_path}"').fetchone()[0] > 0:
        print(f'WARNING: Image {image_path} is already in the database. Skipping to next image.')
        continue
        
    # Detect objects in image
    results_gpu = rs.run_sam3_semantic_predictor(input_image_path=image_path, text_prompts=text_prompts)
    
    # Free up GPU memory in preparation for detecting objects in the next image
    # This is a work-around to prevent out-of-memory errors from the GPU
    # I move all results for further processing and use the GPU only for object detection.
    ic('copying results_gpu to results_cpu')
    results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
    ic('deleting results_gpu from GPU and clearing caches')       
    del results_gpu 
    gc.collect() 
    torch.cuda.empty_cache() # Clears unoccupied cached memory
   
    # populate 'images' table
    image_width, image_height, timestamp, latitude, longitude = rs.get_data_for_images_table(results_cpu, image_path)
    sql = """
    INSERT INTO images (image_path, image_width, image_height, timestamp, latitude, longitude) 
    VALUES (?,?,?,?,?,?)
    RETURNING image_id
    """
    parameters = (image_path, image_width, image_height, timestamp, latitude, longitude,)
    
    # sql = 'INSERT INTO images (image_path) VALUES (?) RETURNING image_id'
    try:
        image_id = con.execute(sql, parameters).fetchone()[0] # THE COMMA IN THE PARAMETERS TUPLE IS IMPORTANT
    except sqlite3.IntegrityError as e:
        print(f'ERROR: Image {image_path} already exists in {db_path}')
        raise e    
    con.commit()
    ic(image_id)
    
    # populate 'detections' table
    # df_detections = rs.get_data_for_detections_table(results_cpu, image_id)
    df_detections = rs.get_data_for_detections_table(results_cpu, image_id)
    for i, r in df_detections.iterrows():
            # populate 'detections' table
        sql = ''' 
        INSERT INTO detections
            (image_id, class_id, poly_wkt, poly_wkt_c, x_min, y_min, x_max, y_max, confidence)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
        '''
        parameters = (image_id, 0, r['poly_wkt'], r['poly_wkt_c'], r['x_min'], r['y_min'], r['x_max'], r['y_max'], r['confidence']) 
        con.execute(sql, parameters)
        con.commit()
        
    # scan the 'detections' table for vcuts and populate 'vcuts' table  
    df_vcuts = rs.get_data_for_vcuts_table(db_path=db_path, image_id=0)
    df_vcuts.to_sql('vcuts', sqlite3.connect(db_path), if_exists='delete_rows', index=False)
    
con.close()   
print('FINISHED')    

CUDA version: 13.0
GPU device name: NVIDIA GeForce RTX 3080 Laptop GPU
Database sam3_detections.sqlite3 created/opened.
SQL script executed successfully.
Changes committed.

Database connection closed.
Ultralytics 8.4.33 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/temp/example_images/08hs-palms-03-zglw-superJumbo.webp: 1932x1932 2 coconut palm trees, 1314.3ms
Speed: 13.4ms preprocess, 1314.3ms inference, 16.8ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict78


ic| 'copying results_gpu to results_cpu'
ic| 'deleting results_gpu from GPU and clearing caches'


ValueError: too many values to unpack (expected 5)

# Retrieve data from database

In [ ]:
pd.read_sql('SELECT * FROM images', sqlite3.connect(db_path))

In [ ]:
pd.read_sql('SELECT * FROM detections', sqlite3.connect(db_path))

In [ ]:
pd.read_sql('SELECT * FROM vcuts', sqlite3.connect(db_path))